# 13 Reproducible Research — Reference Solutions

Complete solutions for the reproducible analysis workflow exercises using the Songbai Nursing Home Legionnaires' disease data.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

## Question 1: Build an outbreak summary dict

In [ ]:
path = Path("data/synthetic/legionella_outbreak.csv")
df = pd.read_csv(path)
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

n_infected = int(df["infected"].sum())
n_deaths = int((df["outcome"] == "dead").sum())

summary = {
    "n_residents": len(df),
    "n_infected": n_infected,
    "n_deaths": n_deaths,
    "attack_rate": f"{df['infected'].mean():.1%}",
    "cfr": f"{n_deaths / n_infected:.1%}",
}

print("=== Outbreak Summary ===")
for k, v in summary.items():
    print(f"  {k}: {v}")

print("\n-> 280 residents, 121 infected, 19 deaths")
print("-> Attack rate 43.2%, CFR 15.7%")

## Question 2: Reproducibility checklist

In [ ]:
checks = {
    "uv.lock exists": Path("uv.lock").exists(),
    "pyproject.toml exists": Path("pyproject.toml").exists(),
    "data file exists": Path("data/synthetic/legionella_outbreak.csv").exists(),
}

print("=== Reproducibility Checklist ===")
for item, ok in checks.items():
    status = "✓" if ok else "✗"
    print(f"  [{status}] {item}")

all_pass = all(checks.values())
print(f"\n-> {'All checks passed!' if all_pass else 'Some checks failed'}")

print("\n=== Why each item matters ===")
print("  uv.lock -> ensures all package versions stay consistent")
print("  pyproject.toml -> defines the project's package requirements")
print("  data file -> no input means no output")

## Question 3 (Challenge): Summary output and verification

In [ ]:
import json
import sys

# Save as CSV
summary_df = pd.DataFrame([summary])
output_path = Path("data/processed")
output_path.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(output_path / "summary.csv", index=False)
print("Saved: data/processed/summary.csv")

# Read back and verify
reloaded = pd.read_csv(output_path / "summary.csv")
print(f"\n=== Verification ===")
print(f"Original n_residents: {summary['n_residents']}")
print(f"Reloaded n_residents: {reloaded['n_residents'].iloc[0]}")
print(f"Match: {summary['n_residents'] == reloaded['n_residents'].iloc[0]}")

# Version info
print(f"\n=== Environment Versions ===")
print(f"  Python: {sys.version.split()[0]}")
print(f"  pandas: {pd.__version__}")
print(f"  numpy: {np.__version__}")

print("\n=== Factors that could cause different results ===")
print("  1. Different package versions (e.g., pandas behavior changes)")
print("  2. Different Python version")
print("  3. Data file modified or missing")
print("  4. Randomness used without a fixed seed")
print("  5. Operating system differences (floating-point precision)")
print("\n-> uv.lock + git solves the first 3 problems")

### Interpretation

- The **summary dict** is the minimal verifiable unit — anyone who runs it should get 280 residents, 121 infected, 19 deaths
- The **checklist** ensures the environment is complete; missing any single item can make the analysis impossible to reproduce
- **Version records** are key to debugging — if results differ, compare versions first
- **The three pillars of reproducibility**: fixed data + version-controlled code + a locked environment

## Question 4: Fix the random seed for reproducibility (dengue fever scenario)

1. Use `np.random.default_rng(seed)` with the same seed to generate simulated dengue case counts twice (20 days each)
2. Use `np.array_equal` to verify the two runs are identical
3. Generate once more with a different seed and explain why the result differs

In [ ]:
import numpy as np
a = np.random.default_rng(42).poisson(5, 20)
b = np.random.default_rng(42).poisson(5, 20)
print("Seed 42, two runs match:", np.array_equal(a, b))
c = np.random.default_rng(99).poisson(5, 20)
print("Seed 99 vs seed 42 match:", np.array_equal(a, c))
print("Interpretation: same seed -> same random sequence -> reproducible results; a different seed gives a different sequence.")

## Question 5: Hashing a data file (COVID-19 scenario)

1. Build a small COVID-19 summary DataFrame
2. Use `hashlib.md5` to compute a hash of its CSV string
3. Explain why a hash can be used to verify whether data has been altered

In [ ]:
import pandas as pd, hashlib
df = pd.DataFrame({"region": ["北", "南", "東", "西"], "cases": [120, 60, 45, 30]})
csv_str = df.to_csv(index=False)
digest = hashlib.md5(csv_str.encode("utf-8")).hexdigest()
print("MD5 =", digest)
print("Interpretation: if even a single cell in the data changes, the hash changes -> hashes can verify data integrity / detect tampering.")

## Question 6: Export a summary to JSON and verify it (measles scenario)

1. Build a measles outbreak summary `dict` (cases, coverage, VE)
2. Save it with `json.dump`, then read it back with `json.load`
3. Use `==` to verify the reloaded content exactly matches the original

In [ ]:
import json
summary = {"cases": 37, "coverage_pct": 88.5, "ve_pct": 93.8}
with open("measles_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False)
with open("measles_summary.json", encoding="utf-8") as f:
    loaded = json.load(f)
print("Reloaded content =", loaded)
print("Matches original:", loaded == summary)

## Question 7: A reproducible analysis function (norovirus scenario)

1. Define a function `attack_rate(cases, population)` with a docstring
2. The function must validate its input (population > 0) and return the attack rate
3. Test it with norovirus banquet numbers and print the result

In [ ]:
def attack_rate(cases, population):
    """Attack rate = number of cases / population; population must be > 0, otherwise raises ValueError."""
    if population <= 0:
        raise ValueError("population must be greater than 0")
    return cases / population

ar = attack_rate(48, 150)
print(f"Norovirus banquet attack rate = {ar:.1%}")

## Question 8 (Challenge): A mini reproducible analysis pipeline (tuberculosis scenario)

Chain together a complete reproducible workflow:
1. Fix a seed to generate synthetic tuberculosis data (with age, smear_positive, cured)
2. Compute the cure rate and save it as JSON
3. Read the JSON back and verify the numbers match
4. Explain how "seed + version + output hash" lets others reproduce the analysis

In [ ]:
import numpy as np, json
rng = np.random.default_rng(2026)
n = 300
age = rng.integers(18, 85, n)
smear = rng.binomial(1, 0.4, n)
cured = rng.binomial(1, 0.75 - 0.1 * smear, n)
result = {"n": int(n), "cured_rate": round(float(cured.mean()), 4)}
with open("tb_repro.json", "w") as f:
    json.dump(result, f)
with open("tb_repro.json") as f:
    back = json.load(f)
assert back["cured_rate"] == result["cured_rate"]
print("Cured rate =", result["cured_rate"], "| Reproducible pipeline complete")
print("Interpretation: a fixed seed (same data) + recorded package versions + output hash/JSON let others rerun the analysis and get the same numbers.")